<a href="https://colab.research.google.com/github/prithwis/parashar21/blob/main/TextCleaner.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import re
from collections import Counter

input_file = "HM_Clean_00.txt"
output_file = "HM_wordcount.txt"

with open(input_file, "r", encoding="cp1252") as f:
#with open(input_file, "r", encoding="utf-8") as f:
    text = f.read()

words = re.findall(r"\b[A-Za-z]+\b", text)
counts = Counter(word.lower() for word in words)

with open(output_file, "w", encoding="utf-8") as f:
    for word in sorted(counts):
        if counts[word] >= 2:
            f.write(f"{word:<25} {counts[word]:>6}\n")

print(f"Done. {len(counts):,} unique words written to {output_file}")

Done. 5,904 unique words written to HM_wordcount.txt


In [ ]:
!pip -q install wordfreq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.8/56.8 MB 17.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 183.1/183.1 kB 13.1 MB/s eta 0:00:00


In [ ]:
import re
from collections import Counter
from wordfreq import zipf_frequency

input_file = "HM_Clean_00.txt"
output_file = "HM_special_wordcount.txt"

# Read Harihar
with open(input_file, "r", encoding="cp1252") as f:
    text = f.read()

# Extract words
words = re.findall(r"\b[A-Za-z]+\b", text)
counts = Counter(word.lower() for word in words)

with open(output_file, "w", encoding="utf-8") as f:

    for word in sorted(counts):

        # Ignore very rare words
        if counts[word] < 2:
            continue

        # Ignore single-letter OCR rubbish
        if len(word) < 2:
            continue

        # If wordfreq recognises it as a reasonably
        # common English word, throw it away
        if zipf_frequency(word, "en") >= 2.5:
            continue

        f.write(f"{word:<30} {counts[word]:>6}\n")

print("Done.")
print("Output:", output_file)

Done.
Output: HM_special_wordcount.txt


In [ ]:
word = "pusa"   # change this to the suspicious word

found = False

for lineno, line in enumerate(text.splitlines(), 1):
    if word.lower() in line.lower():
        print(f"Line {lineno}:")
        print(repr(line))
        found = True

if not found:
    print("NOT FOUND:", word)

Line 211:
"   Punarvasu Nakshatra: Spread from 20° degree Mithune upto 3°-20' Karkata. Presiding deity 'Aditi', the Lord is Budha and Chandra. Symbol - Quiver ( receptacle for arrows). The word Punarvasu is derived from Puna + Vasu, which means return, renewal, restoration or repetition. The 12 Adityas were born of Kasyapa in the womb of Aditi. The 12 are Indra, Vishnu, Vaga, Twasta, Barun, Aryama, Pusa., Mitra, Agni, Parjyanya, Vivaswan and Dinakar. The mother Aditi of whom the Gods are born is the repository of everything good - truth, generosity, magnanimity, purity, aristocracy, beauty and renown. It follows that this start is the cause for these virtues. To start afresh after having once broken off, to start a new life, to come back from a distant land - all these are signified by Punarvasu. It stands for freedom from restriction and limitation, and boundless space. The Gods. the children of Aditi, are basically and essentially are different from children of Diti, who are demons. 

In [ ]:
import re
import nltk
from collections import Counter
from nltk.corpus import words as nltk_words

# Download dictionary if necessary
nltk.download("words", quiet=True)

input_file = "HM_Clean_00.txt"
output_file = "HM_Special_Wordcount.txt"

# English dictionary
english_words = {w.lower() for w in nltk_words.words()}

# HM_Clean_00 is the clean starting corpus
# If this gives a UnicodeDecodeError, change utf-8 to cp1252
with open(input_file, "r", encoding="cp1252") as f:
    text = f.read()

# Extract alphabetic tokens only.
# Thus "Pusa." becomes "pusa", which is intentional.
tokens = re.findall(r"\b[A-Za-z]+\b", text)

# Case-insensitive frequency count
counts = Counter(token.lower() for token in tokens)

special = []

for word, count in counts.items():

    # Ignore words of 2 characters or less
    if len(word) <= 2:
        continue

    # Ignore words occurring only once
    if count < 2:
        continue

    # Ignore recognised English dictionary words
    if word in english_words:
        continue

    special.append((word, count))

# Alphabetical order
special.sort()

with open(output_file, "w", encoding="utf-8") as f:
    for word, count in special:
        f.write(f"{word:<30} {count:>6}\n")

print(f"Total tokens in corpus : {len(tokens):,}")
print(f"Unique tokens          : {len(counts):,}")
print(f"Special words retained : {len(special):,}")
print(f"Written to             : {output_file}")

Total tokens in corpus : 74,588
Unique tokens          : 5,904
Special words retained : 830
Written to             : HM_Special_Wordcount.txt


In [ ]:
import re
import nltk
from collections import Counter
from nltk.corpus import words as nltk_words

# --------------------------------------------------
# HM SIEVE 01
# Find recurring non-English / unusual words
# --------------------------------------------------

nltk.download("words", quiet=True)

input_file = "HM_Clean_01.txt"
output_file = "HM_Sieve_01a.txt"

# English dictionary
english_words = {w.lower() for w in nltk_words.words()}

# Read corpus
# If utf-8 fails, change encoding to "cp1252"
with open(input_file, "r", encoding="cp1252") as f:
    text = f.read()

# Extract alphabetic words only and convert to lowercase
tokens = re.findall(r"\b[A-Za-z]+\b", text)
counts = Counter(word.lower() for word in tokens)

special = []

for word, count in counts.items():

    # Ignore very short tokens
    if len(word) <= 2:
        continue

    # Ignore one-off occurrences
    if count < 2:
        continue

    # Ignore recognised English words
    if word in english_words:
        continue

    special.append((word, count))

# Alphabetical order
special.sort()

with open(output_file, "w", encoding="utf-8") as f:
    for word, count in special:
        f.write(f"{word:<30} {count:>6}\n")

print(f"Total tokens          : {len(tokens):,}")
print(f"Unique tokens         : {len(counts):,}")
print(f"Sieve 01 candidates   : {len(special):,}")
print(f"Written to            : {output_file}")

Total tokens          : 74,490
Unique tokens         : 5,826
Sieve 01 candidates   : 788
Written to            : HM_Sieve_01a.txt


In [ ]:
import re
from collections import Counter

# --------------------------------------------------
# HM SIEVE 02
# Find probable OCR errors by near-neighbour matching
# --------------------------------------------------

input_file = "HM_Clean_01.txt"
output_file = "HM_Sieve_02a.txt"

# Read corpus
# If utf-8 fails, change encoding to "cp1252"
with open(input_file, "r", encoding="cp1252") as f:
    text = f.read()

# Extract alphabetic words and normalise case
tokens = re.findall(r"\b[A-Za-z]+\b", text)
counts = Counter(word.lower() for word in tokens)


# --------------------------------------------------
# Levenshtein distance
# --------------------------------------------------

def edit_distance(a, b):

    # Quick rejection
    if abs(len(a) - len(b)) > 1:
        return 99

    previous = list(range(len(b) + 1))

    for i, ca in enumerate(a, start=1):

        current = [i]

        for j, cb in enumerate(b, start=1):

            insertion = current[j - 1] + 1
            deletion = previous[j] + 1
            substitution = previous[j - 1] + (ca != cb)

            current.append(min(insertion, deletion, substitution))

        previous = current

    return previous[-1]


# --------------------------------------------------
# Candidate selection
# --------------------------------------------------

# Suspect words:
# occur at least twice, but no more than 20 times
suspects = [
    word for word, count in counts.items()
    if len(word) > 2
    and 2 <= count <= 20
]

# Possible correct words:
# occur at least 10 times
reference_words = [
    word for word, count in counts.items()
    if len(word) > 2
    and count >= 10
]

results = []

for suspect in suspects:

    for reference in reference_words:

        if suspect == reference:
            continue

        # Only compare words of almost identical length
        if abs(len(suspect) - len(reference)) > 1:
            continue

        # Reference should be substantially more common
        if counts[reference] < counts[suspect] * 3:
            continue

        # Exactly one insertion/deletion/substitution away
        if edit_distance(suspect, reference) == 1:

            results.append(
                (
                    suspect,
                    counts[suspect],
                    reference,
                    counts[reference]
                )
            )


# Sort by suspect word, then strongest reference candidate
results.sort(key=lambda x: (x[0], -x[3]))


# --------------------------------------------------
# Write report
# --------------------------------------------------

with open(output_file, "w", encoding="utf-8") as f:

    f.write(
        f"{'SUSPECT':<25}"
        f"{'COUNT':>8}    "
        f"{'POSSIBLE CORRECTION':<25}"
        f"{'COUNT':>8}\n"
    )

    f.write("-" * 72 + "\n")

    for suspect, scount, reference, rcount in results:

        f.write(
            f"{suspect:<25}"
            f"{scount:>8}    "
            f"{reference:<25}"
            f"{rcount:>8}\n"
        )


print(f"Total tokens          : {len(tokens):,}")
print(f"Unique tokens         : {len(counts):,}")
print(f"Low-frequency suspects: {len(suspects):,}")
print(f"Sieve 02 candidate pairs: {len(results):,}")
print(f"Written to            : {output_file}")

Total tokens          : 74,490
Unique tokens         : 5,826
Low-frequency suspects: 2,540
Sieve 02 candidate pairs: 452
Written to            : HM_Sieve_02a.txt


In [12]:
import csv
import re

INPUT_FILE   = "HM_Clean_04.txt"
REPLACE_FILE = "replace5.txt"
OUTPUT_FILE  = "HM_Clean_05.txt"

# ------------------------------------------------------------
# 1. Load replacement table
#    Format: wrong,right,category
#    The category column is deliberately ignored.
# ------------------------------------------------------------

replacements = {}

with open(REPLACE_FILE, "r", encoding="utf-8-sig") as f:
    reader = csv.DictReader(f)

    for row in reader:
        wrong = row["wrong"].strip()
        right = row["right"].strip()

        if wrong and right:
            replacements[wrong] = right

print(f"Loaded {len(replacements)} replacement rules.")


# ------------------------------------------------------------
# 2. Read HM_Clean_00
# ------------------------------------------------------------

with open(INPUT_FILE, "r", encoding="cp1252") as f:
    text = f.read()


# ------------------------------------------------------------
# 3. Replace whole words only
#    Case-insensitive, but preserve the replacement exactly
#    as specified in replace2.txt.
# ------------------------------------------------------------

total_replacements = 0

for wrong, right in replacements.items():

    pattern = r"\b" + re.escape(wrong) + r"\b"

    text, count = re.subn(
        pattern,
        right,
        text,
        flags=re.IGNORECASE
    )

    if count > 0:
        print(f"{wrong:20s} -> {right:20s} : {count}")

    total_replacements += count


# ------------------------------------------------------------
# 4. Write HM_Clean_01
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(text)


print("\n----------------------------------------")
print(f"Total replacements : {total_replacements}")
print(f"Output written to  : {OUTPUT_FILE}")
print("----------------------------------------")

Loaded 68 replacement rules.
adra                 -> ardra                : 11
visaka               -> visakha              : 10
sravana              -> shravana             : 6
visakha              -> visaka               : 10

----------------------------------------
Total replacements : 37
Output written to  : HM_Clean_05.txt
----------------------------------------


In [10]:
import re
from collections import Counter

INPUT_FILE = "HM_Clean_05.txt"
OUTPUT_FILE = "HM_Number_Sieve2.txt"

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read()

# ------------------------------------------------------------
# Look for tokens potentially representing 10, 10th, 11, 11th
#
# OCR confusions:
#   1 <-> I <-> i <-> l
#   0 <-> O <-> o
#
# Examples:
#   10, 1O, IO, I0, l0, lO
#   10th, IOth, lOth
#   11, II, I1, 1I, ll
#   11th, IIth, I Ith, l1th, etc.
# ------------------------------------------------------------

patterns = [
    # Possible 10 / 10th
    r'\b[1Il][0Oo](?:th)?\b',

    # Possible 11 / 11th, including an OCR-created space
    r'\b[1Il]\s*[1Il](?:th)?\b',
]

matches = []

for pattern in patterns:
    matches.extend(re.findall(pattern, text, flags=re.IGNORECASE))

counts = Counter(matches)

# Sort case-insensitively
results = sorted(counts.items(), key=lambda x: x[0].lower())

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write("SUSPECT\tCOUNT\n")
    f.write("--------------------\n")

    for token, count in results:
        f.write(f"{token}\t{count}\n")

print("Possible 10/11 OCR forms found:\n")

for token, count in results:
    print(f"{repr(token):15s} : {count}")

print(f"\nOutput written to: {OUTPUT_FILE}")

Possible 10/11 OCR forms found:

'10'            : 37
'10th'          : 208
'11'            : 39
'11th'          : 136
'II'            : 1
'ii'            : 48
'LL'            : 1

Output written to: HM_Number_Sieve2.txt


In [3]:
import re

INPUT_FILE = "HM_Clean_01.txt"

# Suspicious forms found by the numeric sieve
suspects = [
    "I Ith",
    "l0",
    "l0th",
    "llth",
    "lo",
    "II",
    "LL"
]

# Longer forms first, to avoid partial matching
suspects.sort(key=len, reverse=True)

pattern = re.compile(
    r'\b(?:' + '|'.join(re.escape(x) for x in suspects) + r')\b'
)

print("SUSPICIOUS NUMERIC/OCR FORMS")
print("=" * 80)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):

        matches = pattern.findall(line)

        if matches:
            print(f"\nLINE {line_no}   FOUND: {', '.join(matches)}")
            print(line.rstrip())

print("\n" + "=" * 80)
print("Done.")

SUSPICIOUS NUMERIC/OCR FORMS

LINE 5   FOUND: LL
B. A. LL. B. (Ca/.), A. C. I. I. (Lond,)

LINE 128   FOUND: II
CHAPTER- II

Done.


In [13]:
import re
from collections import Counter
from difflib import SequenceMatcher

INPUT_FILE = "HM_Clean_05.txt"   # change if latest file has another name
OUTPUT_FILE = "HM_Nakshatra_Audit4.txt"

nakshatras = [
    "ashwini",
    "bharani",
    "krittika",
    "rohini",
    "mrigashira",
    "ardra",
    "punarvasu",
    "pushya",
    "ashlesha",
    "magha",
    "purvaphalguni",
    "uttaraphalguni",
    "hasta",
    "chitra",
    "swati",
    "visakha",
    "anuradha",
    "jyestha",
    "mula",
    "purvashadha",
    "uttarashadha",
    "shravana",
    "dhanistha",
    "shatabhisha",
    "purvabhadra",
    "uttarabhadrapada",
    "revati"
]

# ------------------------------------------------------------
# Read corpus and count words
# ------------------------------------------------------------

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read().lower()

words = re.findall(r"[a-z]+", text)
counts = Counter(words)

# Only reasonably substantial words
vocabulary = {
    word: count
    for word, count in counts.items()
    if len(word) >= 4
}

# ------------------------------------------------------------
# Find words resembling each canonical nakshatra
# ------------------------------------------------------------

THRESHOLD = 0.55

with open(OUTPUT_FILE, "w", encoding="utf-8") as out:

    for nak in nakshatras:

        candidates = []

        for word, count in vocabulary.items():

            ratio = SequenceMatcher(None, nak, word).ratio()

            if ratio >= THRESHOLD:
                candidates.append((ratio, word, count))

        candidates.sort(reverse=True)

        heading = f"\n=== {nak.upper()} ===\n"

        print(heading.strip())
        out.write(heading)

        # Show best 15 candidates
        for ratio, word, count in candidates[:15]:

            line = (
                f"{word:25s} "
                f"count={count:<5d} "
                f"similarity={ratio:.2f}"
            )

            print(line)
            out.write(line + "\n")

print(f"\nAudit written to: {OUTPUT_FILE}")

=== ASHWINI ===
aswini                    count=12    similarity=0.92
aswi                      count=2     similarity=0.73
shining                   count=3     similarity=0.71
dashing                   count=1     similarity=0.71
swinging                  count=1     similarity=0.67
shown                     count=1     similarity=0.67
shine                     count=1     similarity=0.67
fashions                  count=1     similarity=0.67
shrine                    count=1     similarity=0.62
rohini                    count=13    similarity=0.62
signify                   count=9     similarity=0.57
shrines                   count=3     similarity=0.57
shivaji                   count=1     similarity=0.57
passion                   count=4     similarity=0.57
lakshmi                   count=2     similarity=0.57
=== BHARANI ===
bharani                   count=5     similarity=1.00
bharavi                   count=2     similarity=0.86
varani                    count=1     similarity=0

In [17]:
import re

INPUT_FILE  = "HM_Clean_06.txt"       # change if your latest file has another name
OUTPUT_FILE = "HM_Sieve_03_Encoding3.txt"

# Characters commonly produced by broken UTF-8 / Windows-1252 decoding
suspicious_chars = [
    "Ã", "Â", "Æ", "â", "€", "™", "œ", "ž",
    "ƒ", "‚", "Å", "¤", "¢", "¬", "§", "�"
]

# Build regex
pattern = re.compile("|".join(re.escape(x) for x in suspicious_chars))

hits = []

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    for line_no, line in enumerate(f, start=1):
        found = pattern.findall(line)

        if found:
            hits.append((line_no, found, line.rstrip()))

# Write output
with open(OUTPUT_FILE, "w", encoding="utf-8") as out:
    out.write("ENCODING / MOJIBAKE SIEVE\n")
    out.write("=" * 100 + "\n\n")

    for line_no, found, line in hits:
        chars = " ".join(sorted(set(found)))

        out.write(f"LINE {line_no}   SUSPECT: {chars}\n")
        out.write(line + "\n")
        out.write("-" * 100 + "\n")

# Console summary
print("=" * 60)
print("ENCODING / MOJIBAKE SIEVE")
print("=" * 60)
print(f"Lines flagged : {len(hits)}")
print(f"Output file   : {OUTPUT_FILE}")
print("=" * 60)

ENCODING / MOJIBAKE SIEVE
Lines flagged : 0
Output file   : HM_Sieve_03_Encoding3.txt


In [15]:
# HM_Clean_Encoding.py
#
# Repairs known mojibake / encoding corruption in the Harihar corpus.
# ONLY exact known sequences are replaced.
# No general Unicode manipulation is performed.

INPUT_FILE  = "HM_Clean_05.txt"     # <-- change to your latest file
OUTPUT_FILE = "HM_Clean_06.txt"


# ------------------------------------------------------------
# Exact mojibake replacements
# ------------------------------------------------------------

replacements = {

    # Degree symbol
    "ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â°": "°",

    # Soft hyphen produced by line-breaking / encoding corruption.
    # Delete it so:
    # ideal­istic     -> idealistic
    # counter­balance -> counterbalance
    # learn­ing       -> learning
    "ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â­": "",

    # Bullet
    "ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â¢": "•",

    # Middle dot / stray separator.
    # In this OCR corpus it appears largely as an unwanted artifact.
    # Replace with a space rather than preserving the dot.
    "ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â·": " ",
}


# ------------------------------------------------------------
# Read source
# ------------------------------------------------------------

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    text = f.read()


# ------------------------------------------------------------
# Apply exact replacements
# ------------------------------------------------------------

print("ENCODING CLEANUP")
print("=" * 70)

total = 0

for wrong, right in replacements.items():

    count = text.count(wrong)

    if count:
        display_right = repr(right)

        print(
            f"{repr(wrong):45s} -> "
            f"{display_right:8s} : {count}"
        )

        text = text.replace(wrong, right)
        total += count


# ------------------------------------------------------------
# Write cleaned file
# ------------------------------------------------------------

with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
    f.write(text)


print("=" * 70)
print(f"Total replacements : {total}")
print(f"Output written to  : {OUTPUT_FILE}")
print("=" * 70)


# ------------------------------------------------------------
# Safety check
# Look for the characteristic mojibake characters again.
# ------------------------------------------------------------

suspicious = [
    "Ã", "Â", "Æ", "â", "€", "™", "œ", "ž",
    "ƒ", "‚", "Å", "¤", "¢", "¬", "§", "�"
]

remaining = {}

for char in suspicious:
    count = text.count(char)
    if count:
        remaining[char] = count


print("\nPOST-CLEANING CHECK")
print("=" * 70)

if not remaining:
    print("No characteristic mojibake characters remain.")
else:
    print("WARNING: suspicious characters still remain:")
    for char, count in remaining.items():
        print(f"{repr(char):10s} : {count}")

print("=" * 70)

ENCODING CLEANUP
'ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â°'                        -> '°'      : 62
'ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â\xad'                     -> ''       : 27
'ÃƒÆ’Ã‚Â¢ÃƒÂ¢Ã¢â‚¬Å¡Ã‚Â¬Ãƒâ€šÃ‚Â¢'            -> '•'      : 6
'ÃƒÆ’Ã¢â‚¬Å¡Ãƒâ€šÃ‚Â·'                        -> ' '      : 72
Total replacements : 167
Output written to  : HM_Clean_06.txt

POST-CLEANING CHECK
'Ã'        : 8
'Â'        : 2
'Æ'        : 2
'â'        : 4
'€'        : 2
'ƒ'        : 4
'‚'        : 4
'Å'        : 2
'¢'        : 3
'¬'        : 2
